# AI Session 1 Part 1

### Topic 1: File I/O: JSON


In [11]:
import json

conversation = [
    {"role": "AI", "content": "What do you want me to do?"},
    {"role":"Human","content": "Tell me my todo list"}
]

#dump
with open("conversation.json","w") as f:
    json.dump(conversation, f, indent=2)

#load
record = str()
with open("conversation.json","r") as f:
    record = json.load(f)

print(record)

#loads
raw_string = '{"name": "Alice", "role": "admin"}'
loaded = json.loads(raw_string)
print(loaded["name"])

#dumps
raw = json.dumps(loaded)
print(raw)

[{'role': 'AI', 'content': 'What do you want me to do?'}, {'role': 'Human', 'content': 'Tell me my todo list'}]
Alice
{"name": "Alice", "role": "admin"}


In [13]:
import json
print(json.dumps([1, 2, 3]) == "[1, 2, 3]")

True


### 📊 Topic 2: File I/O: CSV & JSON Lines (JSONL)

In [22]:
import csv, json

#CSV
records = [
    {"question":"What is 2+2?", "answer": "4"},
    {"question":"Capital of Japan?", "answer": "Tokyo"}
]

with open("question.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "answer"])
    writer.writeheader()
    writer.writerows(records)

with open("question.csv", "r", newline="") as f:
    reader = csv.DictReader(f)
    for i in reader:
        print(i["question"],"->",i["answer"])

# JSONL -- one JSON object per line, appended over time

with open("log.jsonl", "w", newline="") as f:
    pass

with open("log.jsonl", "a") as f:
    f.write(json.dumps({"event": "Start"}) + "\n")
    f.write("{\"event\": \"end\"}\n")

with open("log.jsonl", "r") as f:
    for line in f:
        print(json.loads(line))



What is 2+2? -> 4
Capital of Japan? -> Tokyo
{'event': 'Start'}
{'event': 'end'}


### Topic 3: Environment Variables & .env Secrets

In [27]:
from dotenv import load_dotenv
import os

with open("API.env", "w") as f:
    f.write("DEMO_API_KEY=gsk_demo_key_not_real_1234567890\n")

load_dotenv("API.env")

api = os.getenv("DEMO_API_KEY")
print(f"Key loaded, starts with: {api[:8]}...")   # never print the full key

Key loaded, starts with: gsk_demo...


### 🌐 Topic 4: Calling REST APIs with `requests`

In [32]:
import requests

response = requests.get("https://api.github.com/users/octocat", timeout = 10)
print(response.status_code)

if response.status_code == 200:
    data = response.json()
    print(data["login"],":", data["public_repos"])
else:
    print(f"Request did not succeed (status {response.status_code}) -- this is EXACTLY why "
          "you always check status_code before trusting the body, even for 'reliable' public APIs.")

200
octocat : 8


Making our own server for practise:

In [33]:
import threading, http.server, json

class PracticeHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/status":
            self._respond(200, {"status": "ok", "service": "practice-api"})
        elif self.path == "/notfound-demo":
            self._respond(404, {"error": "not found"})
        else:
            self._respond(200, {"path": self.path, "message": "hello from GET"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length)) if length else {}
        self._respond(201, {"created": True, "you_sent": body})

    def _respond(self, code, payload):
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(payload).encode())

    def log_message(self, *args):
        pass   # silence default request logging for a cleaner notebook

_server = http.server.HTTPServer(("localhost", 0), PracticeHandler)
PRACTICE_PORT = _server.server_address[1]
_thread = threading.Thread(target=_server.serve_forever, daemon=True)
_thread.start()
PRACTICE_URL = f"http://localhost:{PRACTICE_PORT}"
print(f"Local practice API running at {PRACTICE_URL}")

Local practice API running at http://localhost:63047


In [34]:
import requests
response = requests.get(f"{PRACTICE_URL}/status", timeout=5)
print(response.status_code, response.json())

200 {'status': 'ok', 'service': 'practice-api'}


In [35]:
# Example 4: POST request with a JSON body
import requests
payload = {"title": "New task", "completed": False}
response = requests.post(f"{PRACTICE_URL}/tasks", json=payload, timeout=5)
if response.status_code == 201:
    print("Created:", response.json())
else:
    print(f"Failed with status {response.status_code}: {response.text}")

Created: {'created': True, 'you_sent': {'title': 'New task', 'completed': False}}


### 🧠 Topic 5: LLM Concepts: Tokens, Context Windows & Roles

In [ ]:
def estimate_tokens(text):
    """Rough estimate: 1 token ~= 4 characters."""
    return len(text) // 4

sample = "Explain what a REST API is in two sentences."
print(f"'{sample}'")
print(f"Estimated tokens: {estimate_tokens(sample)}")

messages = [
    {"role": "system", "content": "You are a concise, factual assistant."},
    {"role": "user", "content": "What is the boiling point of water at sea level?"},
    {"role": "assistant", "content": "100°C (212°F) at standard atmospheric pressure."},
    {"role": "user", "content": "And on Mount Everest?"},
]
for m in messages:
    print(f"[{m['role']}] {m['content']}")

In [36]:
# Example 2: Longer text = more tokens = higher cost -- a direct relationship
def estimate_tokens(text):
    return len(text) // 4

short_prompt = "Summarize this."
long_prompt = "Summarize this article in detail, covering every major point, subpoint, and nuance discussed by the author across all sections."
print(f"Short: ~{estimate_tokens(short_prompt)} tokens")
print(f"Long: ~{estimate_tokens(long_prompt)} tokens")

Short: ~3 tokens
Long: ~31 tokens


In [37]:
# Example 4: Checking whether a conversation fits inside a context window budget
def estimate_tokens(text):
    return len(text) // 4

def fits_in_context(messages, max_tokens):
    total = sum(estimate_tokens(m["content"]) for m in messages)
    return total <= max_tokens

messages = [{"role": "user", "content": "a" * 100}]
print(fits_in_context(messages, max_tokens=20))    # False -- ~25 tokens estimated, over budget
print(fits_in_context(messages, max_tokens=50))    # True -- fits comfortably

False
True


### 🚀 Topic 6: Calling a Real LLM API — Free with Groq

In [4]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("API.env", override=True)
client = Groq(api_key=os.getenv("Groq_Key"))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    max_tokens=500,
    messages=[
        {"role": "system", "content": "You are a concise, factual assistant."},
        {"role": "user", "content": "Explain what a REST API is in two sentences."},
    ],
)
print(response.choices[0].message.content)



REST is an architectural style for web services that relies on stateless, client‑server communication—typically using HTTP. It uses standard HTTP methods (GET, POST, PUT, DELETE, etc.) to perform CRUD operations on resources identified by URLs.


In [8]:
import os
import json
import time
import argparse
from dotenv import load_dotenv

def estimate_tokens(text):
    """Rough estimate: 1 token ~= 4 characters."""
    return len(text) // 4

def load_client():
    """Load .env and initialize a Groq client. Raise a clear error if the key is missing."""
    load_dotenv("API.env")
    api_key = os.getenv("Groq_Key")
    if not api_key:
        raise ValueError(
            "GROQ_API_KEY not found -- get a free key at console.groq.com "
            "and add it to a .env file as GROQ_API_KEY=your_key_here"
        )
    from groq import Groq
    return Groq(api_key=api_key)

def get_system_prompt(mode):
    """Return an appropriate system prompt string based on the active mode."""
    if mode == "summarize":
        return "You are a summarization assistant that produces exactly 3 concise bullet points."
    elif mode == "qa":
        return "You are a concise, helpful assistant. Keep answers under 3 sentences."
    else:
        return "You are a helpful assistant."

def call_model_with_retry(client, model, system_prompt, history, max_retries=3, base_delay=1):
    """
    Call the LLM with the system prompt and message history.
    Includes exponential backoff for robust error handling.
    """
    if client is None:
        return f"[SIMULATED -- no live client] Reply to: {history[-1]['content'][:50]}..."
        
    full_messages = [{"role": "system", "content": system_prompt}] + history
    last_error = None
    
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model, 
                max_tokens=300, 
                messages=full_messages
            )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            print(f"Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                time.sleep(base_delay * (2 ** (attempt - 1)))
                
    return f"[Error calling the model after {max_retries} attempts: {last_error}]"

def save_session(history, filepath="session_log.json"):
    """Write the full history list to a JSON file, nicely indented."""
    if not history:
        return
    with open(filepath, "w") as f:
        json.dump(history, f, indent=2)
    print(f"\nSession saved successfully to {filepath}")

def main():
    # Stretch Goal: Add a --mode toggle
    parser = argparse.ArgumentParser(description="LLM-Powered CLI Tool")
    parser.add_argument(
        "--mode", 
        choices=["qa", "summarize"], 
        default="qa", 
        help="Choose assistant mode: 'qa' or 'summarize'"
    )
    import sys
    # If running in Jupyter, ignore the system args and use defaults
    if 'ipykernel' in sys.modules:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()

    try:
        client = load_client()
        print("Client loaded successfully.")
    except ValueError as e:
        client = None
        print(f"(Notice: {e})\nRunning in simulated mode.")

    system_prompt = get_system_prompt(args.mode)
    history = []
    model = "openai/gpt-oss-20b"
    total_tokens = 0

    print(f"\n=== Active Mode: {args.mode.upper()} ===")
    print("Type 'quit' to exit and save the session.")

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (KeyboardInterrupt, EOFError):
            break

        if user_input.lower() in ["quit", "exit"]:
            break
        if not user_input:
            continue

        history.append({"role": "user", "content": user_input})
        total_tokens += estimate_tokens(user_input)

        reply = call_model_with_retry(client, model, system_prompt, history)
        
        history.append({"role": "assistant", "content": reply})
        total_tokens += estimate_tokens(reply)

        print(f"Assistant: {reply}")
        # Stretch Goal: Running token-count estimate
        print(f"[Estimated Session Tokens: {total_tokens}]")

    save_session(history, filepath=f"{args.mode}_session.json")

if __name__ == "__main__":
    main()

Client loaded successfully.

=== Active Mode: QA ===
Type 'quit' to exit and save the session.
Assistant: I’m ChatGPT, a language model developed by OpenAI. I help answer questions and provide information on a wide range of topics.
[Estimated Session Tokens: 34]
Assistant: Python is a high‑level, interpreted language known for its readability, dynamic typing, and extensive standard library, originally created by Guido van Rossum in 1991. It’s widely used for web development, data science, automation, AI, and scripting, supported by a large ecosystem of third‑party packages.
[Estimated Session Tokens: 115]

Session saved successfully to qa_session.json
